# Stage 2 Model Comparison

  
Compares Logistic Regression , Random Forest , and XGBoost  for both Stage 2 targets.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Standardized project root discovery
root = Path.cwd().resolve()
while root != root.parent and not (root / 'README.md').exists():
    root = root.parent
PROJECT_ROOT = root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import sys
from pathlib import Path

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scripts.run_stage2_model_compare import run_model_comparison
from src.models.stage2_xgboost import TARGET_DISPLAY

In [2]:
comparison_df = run_model_comparison()
comparison_df


Comparing models for Unmet Family Planning Need


target_unmet_fp: socioeconomic-only=0.6566 | +barriers=0.6583 | uplift=+0.0017


target_unmet_fp: socioeconomic-only=0.6601 | +barriers=0.6572 | uplift=-0.0030


target_unmet_fp: socioeconomic-only=0.6653 | +barriers=0.6619 | uplift=-0.0034

Saved model comparison -> C:\Users\hireg\OneDrive\Desktop\Major project\Major Phase v2\BarrierLens_MP_G25_P48\outputs\stage2_results\model_comparison_table.csv


Saved ROC-AUC chart -> C:\Users\hireg\OneDrive\Desktop\Major project\Major Phase v2\BarrierLens_MP_G25_P48\outputs\stage2_results\model_comparison_roc_auc.png
Best model for Unmet Family Planning Need: XGBoost (ROC-AUC=0.6679)


,Model,Target,Accuracy,ROC-AUC,Precision,Recall,F1-Score,Baseline_ROC-AUC,Full_ROC-AUC,Barrier_Uplift
0,Logistic Regression,target_unmet_fp,0.5847,0.6591,0.1573,0.6665,0.2546,0.6566,0.6583,0.0017
1,Random Forest,target_unmet_fp,0.6041,0.6672,0.1615,0.6493,0.2587,0.6601,0.6572,-0.0030
2,XGBoost,target_unmet_fp,0.5933,0.6679,0.1597,0.6624,0.2573,0.6653,0.6619,-0.0034


In [3]:
# Best model per target
for target in comparison_df['Target'].unique():
    subset = comparison_df[comparison_df['Target'] == target].sort_values('ROC-AUC', ascending=False)
    best = subset.iloc[0]
    print(f"{TARGET_DISPLAY.get(target, target)}: {best['Model']} (ROC-AUC={best['ROC-AUC']:.4f}, F1={best['F1-Score']:.4f})")

Unmet Family Planning Need: XGBoost (ROC-AUC=0.6679, F1=0.2573)


In [4]:
# ROC-AUC comparison chart
fig, axes = plt.subplots(1, len(comparison_df['Target'].unique()), figsize=(12, 5), squeeze=False)
for ax, target in zip(axes.flat, comparison_df['Target'].unique()):
    subset = comparison_df[comparison_df['Target'] == target]
    sns.barplot(data=subset, x='Model', y='ROC-AUC', ax=ax, palette='viridis')
    ax.set_title(TARGET_DISPLAY.get(target, target))
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
    ax.set_ylim(0.5, 1.0)
plt.tight_layout()
out = PROJECT_ROOT / 'outputs/stage2_results/model_comparison_roc_auc.png'
out.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved -> {out}')

C:\Users\hireg\AppData\Local\Temp\ipykernel_20704\2495840252.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=subset, x='Model', y='ROC-AUC', ax=ax, palette='viridis')
C:\Users\hireg\AppData\Local\Temp\ipykernel_20704\2495840252.py:7: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')


Saved -> C:\Users\hireg\OneDrive\Desktop\Major project\Major Phase v2\BarrierLens_MP_G25_P48\outputs\stage2_results\model_comparison_roc_auc.png


C:\Users\hireg\AppData\Local\Temp\ipykernel_20704\2495840252.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# Barrier uplift by model
display(comparison_df[['Model', 'Target', 'Baseline_ROC-AUC', 'Full_ROC-AUC', 'Barrier_Uplift']])

,Model,Target,Baseline_ROC-AUC,Full_ROC-AUC,Barrier_Uplift
0,Logistic Regression,target_unmet_fp,0.6566,0.6583,0.0017
1,Random Forest,target_unmet_fp,0.6601,0.6572,-0.0030
2,XGBoost,target_unmet_fp,0.6653,0.6619,-0.0034


In [6]:
# Generate PDF report
%run ../scripts/generate_stage2_report.py

FileNotFoundError: Missing C:\Users\hireg\OneDrive\Desktop\Major project\Major Phase v2\BarrierLens_MP_G25_P48\outputs\stage2_results\xgboost_evaluation_results.csv. Run Stage 2 XGBoost first.